In [1]:
import os
import sys
import requests
import datetime as dt
import numpy as np
from dotenv import load_dotenv
from pathlib import Path
import requests
import pandas as pd
import talib as ta
import plotly.graph_objs as go

import ipywidgets as widgets
from ipywidgets import Dropdown, Text, Button, Output
from IPython.display import display

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown

In [2]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()

In [23]:
portfolio = pd.read_csv('鉱業株_調査_20260502.csv')
portfolio["market_code"] = portfolio["market_code"].astype(object)
portfolio["market_code"] = portfolio["market_code"].where(
    portfolio["market_code"].notna(), None
)
# 鉱業株のポートフォリオに、購入価格と保有株数からポジションの価値を計算して追加
portfolio["position_value"] = portfolio["purchase_price_yen"] * portfolio["number_of_shares"]
total_value = portfolio["position_value"].sum()
portfolio["weight"] = portfolio["position_value"] / total_value
# 鉱業株のポートフォリオに、ベータ値を追加（仮にNoneで初期化）
portfolio["beta"] = None
portfolio.head()

,name,code,market_code,positioning,financial_risk_ratio,Budget_yen,acquisition,price,currency_conversion_to_target,number_of_shares,account,status,place,country_risk,dd_flg,exchange_rate_yen,purchase_price_yen,position_value,weight,beta
0,Newmont Corporation Minings,NEM,None,Tire1/Core,LOW,NaN,NaN,58.17,USD,25,SBI,HOLD,NaN,NaN,0,152.00,221046,5526150,0.013079,None
1,Agnico Eagle Mine,AEM,None,Tire1/Core,LOW,NaN,NaN,122.02,USD,25,SBI,HOLD,NaN,NaN,0,152.00,463676,11591900,0.027434,None
2,Wheaton Precious Metals Corp.,WPM,None,Tire1/Core,LOW,NaN,NaN,131.49,USD,5,SBI,HOLD,ー,NaN,0,114.42,75225,376125,0.000890,None
3,Royal Gold Inc,RGLD,None,Tire1/Core,LOW,NaN,NaN,222.74,USD,1,SBI,HOLD,ー,NaN,0,114.42,25486,25486,0.000060,None
4,Pan American Silver Corp,PAAS,None,Tire1/Core,LOW,NaN,NaN,60.00,USD,15,SBI,HOLD,NaN,NaN,0,114.42,102978,1544670,0.003656,None


In [20]:
start = '2025-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

for line in portfolio.itertuples():
    code = line.code
    market = line.market_code
    print(f"Updating {code} ({market})...")
    response = request_api.update_stock_timeseries_data(
        code=code,
        market=market,
        start=start,
        end=end
    )
    print(f"Updated {code} ({market}): {response}")

Updating NEM (None)...
Updated NEM (None): {'result': True}
Updating AEM (None)...
Updated AEM (None): {'result': True}
Updating WPM (None)...
Updated WPM (None): {'result': True}
Updating RGLD (None)...
Updated RGLD (None): {'result': True}
Updating PAAS (None)...
Updated PAAS (None): {'result': True}
Updating AYA (TO)...
Updated AYA (TO): {'result': True}
Updating ITR (V)...
Updated ITR (V): {'result': True}
Updating HL (None)...
Updated HL (None): {'result': True}
Updating ELE (TO)...
Updated ELE (TO): {'result': True}
Updating ALK (AX)...
Updated ALK (AX): {'result': True}
Updating USAS (None)...
Updated USAS (None): {'result': True}
Updating SH (None)...
Updated SH (None): {'result': True}
Updating SICO (V)...
Updated SICO (V): {'result': True}
Updating AGMR (TO)...
Updated AGMR (TO): {'result': True}
Updating UG (CN)...
Updated UG (CN): {'result': True}
Updating PZG (None)...
Updated PZG (None): {'result': True}
Updating ASM (None)...
Updated ASM (None): {'result': True}
Updating

In [ ]:
start = '2025-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

price_df = None  # 最初は None にしておく

for row in portfolio.itertuples():
    if row.status != "HOLD":
        continue

    ts = request_api.get_stock_time_series_data(
        code=row.code,
        market=row.market_code,
        start=start,
        end=end
    )

    ts = ts.set_index("date").sort_index()

    # 銘柄ごとのボラティリティ（年率）
    daily_ret = ts["close"].pct_change().dropna()
    vol = daily_ret.std() * np.sqrt(252)
    portfolio.loc[row.Index, "beta"] = vol

    # price_df の初期化
    if price_df is None:
        price_df = pd.DataFrame(index=ts.index)

    # index を揃えて追加
    price_df[row.code] = ts["close"].reindex(price_df.index)


取得件数: 912
取得件数: 912
取得件数: 384
取得件数: 384
取得件数: 384
取得件数: 562
取得件数: 386
取得件数: 384
取得件数: 386
取得件数: 388
取得件数: 1096


ValueError: cannot reindex on an axis with duplicate labels